In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
TRAIN_PATH = "/content/drive/MyDrive/X_train_sequences.npy"
TEST_PATH  = "/content/drive/MyDrive/X_test_sequences.npy"
SAVE_DIR   = "/content/drive/MyDrive/new_outputs"

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import os

In [4]:
CONFIG = {
    "lstm_hidden"   : 128,    # try 64, 128, 256
    "lstm_layers"   : 2,      # try 1, 2
    "bottleneck"    : 32,     # try 16, 32, 64
    "dropout"       : 0.3,    # regularization
    "learning_rate" : 1e-3,   # try 1e-3, 1e-4
    "weight_decay"  : 1e-5,   # L2 regularization
    "batch_size"    : 64,     # try 32, 64, 128
    "epochs"        : 50,
    "patience"      : 5,     # early stopping
    "val_split"     : 0.1,    # 10% of train for validation
    "seed"          : 42,
}

In [5]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# 1. LOAD DATA
# ─────────────────────────────────────────
X_train = np.load(TRAIN_PATH).astype(np.float32)  # (3588, 10, 128)
X_test  = np.load(TEST_PATH).astype(np.float32)   # (5122, 10, 128)

print(f"Train shape : {X_train.shape}")
print(f"Test  shape : {X_test.shape}")

Train shape : (3588, 10, 128)
Test  shape : (5122, 10, 128)


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Reshape → scale → reshape back
original_train_shape = X_train.shape
original_test_shape  = X_test.shape

X_train = scaler.fit_transform(X_train.reshape(-1, 128)).reshape(original_train_shape)
X_test  = scaler.transform(X_test.reshape(-1, 128)).reshape(original_test_shape)

print("After scaling:")
print(X_train.min(), X_train.max(), X_train.mean(), X_train.std())


After scaling:
-5.517793 8.497102 1.0153388e-08 0.73420876


In [8]:
# Train / Val split
train_tensor = torch.tensor(X_train)
dataset      = TensorDataset(train_tensor)

val_size   = int(len(dataset) * CONFIG["val_split"])
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False)

In [9]:
#  MODEL
# ─────────────────────────────────────────
class LSTM_AE(nn.Module):
    """
    LSTM Autoencoder for temporal anomaly detection.

    Architecture:
        Input (batch, 10, 128)
            ↓
        LSTM  → encodes temporal patterns
            ↓  last hidden state (batch, 128)
        Encoder MLP  → (batch, bottleneck)
            ↓
        Decoder MLP  → (batch, 128)
            ↓
        Repeat across 10 timesteps → (batch, 10, 128)
    """
    def __init__(self, input_size=128, hidden_size=128, num_layers=2,
                 bottleneck=32, dropout=0.3, seq_len=10):
        super().__init__()
        self.seq_len = seq_len
        self.input_size = input_size

        # ── LSTM (temporal modeling) ──────────────────
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0,
            batch_first = True
        )

        # ── Encoder (compress) ────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, bottleneck)
        )

        # ── Decoder (reconstruct) ─────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, seq_len * input_size)  # ← outputs all 10 frames at once
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)

        # LSTM — pass full sequence, take last hidden state
        lstm_out, (h_n, _) = self.lstm(x)
        # h_n: (num_layers, batch, hidden) → take last layer
        last_hidden = h_n[-1]                          # (batch, hidden)

        # Encode → bottleneck
        z = self.encoder(last_hidden)                  # (batch, bottleneck)

        # Decode → reconstruct one frame
        decoded = self.decoder(z)                      # (batch, input_size)

        # Expand to full sequence (repeat across timesteps)
        # NOTE: member doing attention will modify between lstm_out and encoder
        out = decoded.view(-1, self.seq_len, self.input_size) # (batch, 10, 128)

        return out

    def get_lstm_all_hidden(self, x):
        """
        Returns all LSTM hidden states — needed by attention member.
        Shape: (batch, seq_len, hidden_size)
        """
        lstm_out, _ = self.lstm(x)
        return lstm_out


model = LSTM_AE(
    input_size  = 128,
    hidden_size = CONFIG["lstm_hidden"],
    num_layers  = CONFIG["lstm_layers"],
    bottleneck  = CONFIG["bottleneck"],
    dropout     = CONFIG["dropout"],
    seq_len     = 10
).to(device)

print(f"\nModel architecture:\n{model}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


Model architecture:
LSTM_AE(
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3)
  (encoder): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=1280, bias=True)
  )
)
Trainable parameters: 450,080


In [11]:
#  TRAINING
# ─────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = CONFIG["learning_rate"],
    weight_decay = CONFIG["weight_decay"]   # L2 regularization
)
criterion = nn.MSELoss()

train_losses = []
val_losses   = []
best_val_loss = float("inf")
patience_counter = 0

print("\nStarting training...")
for epoch in range(CONFIG["epochs"]):
    # ── Train ──
    model.train()
    epoch_train_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        reconstructed = model(batch)
        loss = criterion(reconstructed, batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()
    epoch_train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for (batch,) in val_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            epoch_val_loss += loss.item()
    epoch_val_loss /= len(val_loader)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | "
          f"Train Loss: {epoch_train_loss:.6f} | "
          f"Val Loss: {epoch_val_loss:.6f}")

    # ── Early stopping ──
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save({
            "epoch"      : epoch,
            "model_state": model.state_dict(),
            "config"     : CONFIG,
            "val_loss"   : best_val_loss
        }, os.path.join(SAVE_DIR, "best_model.pth"))
        print(f"  ✅ Best model saved (val_loss={best_val_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete. Best val loss: {best_val_loss:.6f}")


Starting training...
Epoch   1/50 | Train Loss: 0.250556 | Val Loss: 0.216304
  ✅ Best model saved (val_loss=0.216304)
Epoch   2/50 | Train Loss: 0.215729 | Val Loss: 0.195736
  ✅ Best model saved (val_loss=0.195736)
Epoch   3/50 | Train Loss: 0.197446 | Val Loss: 0.180838
  ✅ Best model saved (val_loss=0.180838)
Epoch   4/50 | Train Loss: 0.190493 | Val Loss: 0.174523
  ✅ Best model saved (val_loss=0.174523)
Epoch   5/50 | Train Loss: 0.183133 | Val Loss: 0.165314
  ✅ Best model saved (val_loss=0.165314)
Epoch   6/50 | Train Loss: 0.182308 | Val Loss: 0.162928
  ✅ Best model saved (val_loss=0.162928)
Epoch   7/50 | Train Loss: 0.177535 | Val Loss: 0.162041
  ✅ Best model saved (val_loss=0.162041)
Epoch   8/50 | Train Loss: 0.177054 | Val Loss: 0.160351
  ✅ Best model saved (val_loss=0.160351)
Epoch   9/50 | Train Loss: 0.174658 | Val Loss: 0.158039
  ✅ Best model saved (val_loss=0.158039)
Epoch  10/50 | Train Loss: 0.172039 | Val Loss: 0.159909
Epoch  11/50 | Train Loss: 0.172151 | V

In [12]:
# 4. COMPUTE RECONSTRUCTION ERRORS
# ─────────────────────────────────────────
# Load best model
checkpoint = torch.load(os.path.join(SAVE_DIR, "best_model.pth"), map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

def compute_errors(data_array, batch_size=64):
    """Returns per-sequence MSE reconstruction error."""
    tensor  = torch.tensor(data_array).to(device)
    dataset = DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=False)
    errors  = []
    with torch.no_grad():
        for (batch,) in dataset:
            recon = model(batch)
            # MSE per sequence: mean over (seq_len, features)
            mse = ((batch - recon) ** 2).mean(dim=(1, 2))
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

print("\nComputing reconstruction errors...")
train_errors = compute_errors(X_train)
test_errors  = compute_errors(X_test)

np.save(os.path.join(SAVE_DIR, "train_reconstruction_errors.npy"), train_errors)
np.save(os.path.join(SAVE_DIR, "test_reconstruction_errors.npy"),  test_errors)

print(f"Train errors saved: shape {train_errors.shape}")
print(f"Test  errors saved: shape {test_errors.shape}")
print("Train errors:", train_errors.min(), train_errors.max(), train_errors.mean())
print("Test errors: ", test_errors.min(),  test_errors.max(),  test_errors.mean())

# Threshold reference (for teammate doing evaluation)
for p in [90, 95, 99]:
    t = np.percentile(train_errors, p)
    print(f"  Threshold @{p}th percentile: {t:.6f}")


Computing reconstruction errors...
Train errors saved: shape (3588,)
Test  errors saved: shape (5122,)
Train errors: 0.004204395 1.0768963 0.14901966
Test errors:  0.004204395 3.3541055 0.39365807
  Threshold @90th percentile: 0.341012
  Threshold @95th percentile: 0.426037
  Threshold @99th percentile: 0.670305


In [13]:
import joblib
joblib.dump(scaler, "/content/drive/MyDrive/outputs/scaler.pkl")

['/content/drive/MyDrive/outputs/scaler.pkl']